# Exploratory look at the feature panel

This notebook reads the cached monthly panel. It does not refit anything.
The universe is the current S&P 500, so every table below embeds survivorship bias.

In [1]:
from pathlib import Path

import pandas as pd

root = Path.cwd()
if not (root / 'pyproject.toml').exists():
    root = root.parent
panel = pd.read_parquet(root / 'data' / 'processed' / 'panel.parquet')
panel['date'] = pd.to_datetime(panel['date'])
print(f"{panel['date'].min().date()} to {panel['date'].max().date()}")
print(f"{panel['date'].nunique()} months, {panel['ticker'].nunique()} tickers, {len(panel):,} rows")
panel.head()

2006-01-31 to 2026-10-31
250 months, 500 tickers, 113,213 rows


,date,ticker,mom_12_1_raw,mom_6_1_raw,mom_3_1_raw,rev_1m_raw,vol_20d_raw,vol_60d_raw,downside_vol_raw,log_dollar_volume_raw,...,vol_20d,vol_60d,downside_vol,log_dollar_volume,amihud,beta_252d,idio_vol,max_daily_ret,fwd_ret,fwd_ret_rank
0,2006-01-31,A,0.505654,0.268674,0.039988,0.018624,0.196792,0.190285,0.115091,18.159262,...,-0.240838,-0.167539,-0.183246,0.382199,-0.549738,0.120419,0.356021,-0.293194,0.061634,0.654450
1,2006-01-31,AAPL,0.869701,0.685581,0.248307,0.050354,0.455337,0.351546,0.200179,21.663054,...,0.848168,0.738220,0.712042,0.989529,-0.984293,0.947644,0.774869,0.748691,-0.092968,-0.895288
2,2006-01-31,ABT,-0.108458,-0.148890,-0.084088,0.101556,0.287425,0.275270,0.196825,18.826565,...,0.387435,0.460733,0.696335,0.685864,-0.670157,-0.837696,-0.141361,0.586387,0.023871,0.136126
3,2006-01-31,ACGL,0.490607,0.190217,0.106061,-0.007489,0.175690,0.205863,0.131276,16.315737,...,-0.481675,-0.047120,0.089005,-0.596859,0.602094,-0.926702,-0.136126,-0.481675,0.040854,0.413613
4,2006-01-31,ACN,0.121670,0.166913,0.097301,0.092137,0.231574,0.199753,0.095264,18.147491,...,0.047120,-0.094241,-0.596859,0.371728,-0.486911,-0.481675,0.209424,0.544503,0.035839,0.361257


## Raw features

The `_raw` columns are the values before the monthly rank normalization.
The unsuffixed columns are the model inputs, in [-1, 1]. `fwd_ret` is next month's
return and is the only column that looks past the signal date. `fwd_ret_rank` is
its cross-sectional rank, also in [-1, 1], and it is the training target.

In [2]:
raw = [column for column in panel.columns if column.endswith('_raw')]
panel[raw].describe().T.round(4)

,count,mean,std,min,25%,50%,75%,max
mom_12_1_raw,113213.0,0.1635,0.4581,-0.9901,-0.0392,0.1251,0.3018,51.9760
mom_6_1_raw,113213.0,0.0703,0.2264,-0.9461,-0.0482,0.0598,0.1689,9.9825
mom_3_1_raw,113213.0,0.0276,0.1341,-0.9102,-0.0430,0.0254,0.0924,2.7349
rev_1m_raw,113213.0,0.0137,0.0935,-0.8435,-0.0357,0.0130,0.0603,2.5966
vol_20d_raw,113213.0,0.2941,0.1998,0.0000,0.1766,0.2430,0.3443,6.4008
vol_60d_raw,113213.0,0.3049,0.1828,0.0000,0.1949,0.2569,0.3531,3.7500
downside_vol_raw,113213.0,0.2049,0.1282,0.0000,0.1263,0.1720,0.2405,2.2778
log_dollar_volume_raw,113208.0,18.6380,1.4324,3.4475,17.9249,18.6948,19.4601,24.8315
amihud_raw,112831.0,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
beta_252d_raw,113213.0,1.0051,0.4342,-1.0948,0.7364,0.9844,1.2445,4.4766


In [3]:
month = panel.loc[panel['date'] == panel['date'].max()].dropna(subset=['mom_12_1'])
print(month['date'].iloc[0].date(), 'names', len(month))
month[['ticker', 'mom_12_1_raw', 'mom_12_1', 'fwd_ret']].sort_values('mom_12_1').head(8)

2026-10-31 names 500


,ticker,mom_12_1_raw,mom_12_1,fwd_ret
112897,FICO,-0.642989,-1.000000,NaN
112830,CSGP,-0.601802,-0.995992,NaN
113084,PODD,-0.583176,-0.991984,NaN
112956,INTU,-0.583104,-0.987976,NaN
112782,BSX,-0.566620,-0.983968,NaN
112752,APP,-0.544302,-0.979960,NaN
112802,CHTR,-0.525744,-0.975952,NaN
113212,ZTS,-0.507779,-0.971944,NaN


A name at the bottom of the momentum rank has a feature near -1.
Appending prices from after this date does not change earlier rows.
That check is `tests/test_features.py`.